<a href="https://colab.research.google.com/github/KK-code001/Candidate-Screening-System/blob/main/rf%2Bxgb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# 1. Load the datasets
train_df = pd.read_csv('train_data.csv')
test_df = pd.read_csv('test_data.csv')

# 2. Preprocessing
train_df['shortlisted'] = train_df['shortlisted'].map({'Yes': 1, 'No': 0})
test_df['shortlisted'] = test_df['shortlisted'].map({'Yes': 1, 'No': 0})

train_df = pd.get_dummies(train_df, columns=['education_level'], drop_first=True)
test_df = pd.get_dummies(test_df, columns=['education_level'], drop_first=True)

# Ensure test columns match training columns
for col in set(train_df.columns) - set(test_df.columns):
    test_df[col] = 0

test_df = test_df[train_df.columns]

# Features and Target
X_train = train_df.drop('shortlisted', axis=1)
y_train = train_df['shortlisted']

X_test = test_df.drop('shortlisted', axis=1)
y_test = test_df['shortlisted']

# ---------------- BASE MODELS ----------------

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42
)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

# ---------------- STACKING MODEL ----------------

stack_model = StackingClassifier(
    estimators=[
        ('rf', rf_model),
        ('xgb', xgb_model)
    ],
    final_estimator=LogisticRegression(),
    cv=5
)

# Train stacking model
stack_model.fit(X_train, y_train)

# Predictions
train_pred = stack_model.predict(X_train)
test_pred = stack_model.predict(X_test)

# Accuracy
train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

print("========== STACKING MODEL (RF + XGBoost) ==========")
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Testing Accuracy: {test_acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

========== STACKING MODEL (RF + XGBoost) ==========
Training Accuracy: 0.9144
Testing Accuracy: 0.8908

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.82      0.82      1832
           1       0.92      0.92      0.92      4168

    accuracy                           0.89      6000
   macro avg       0.87      0.87      0.87      6000
weighted avg       0.89      0.89      0.89      6000

